<a href="https://colab.research.google.com/github/binteaamer/FlyrankInternship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binteaamer/FlyrankInternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**My rule ranks pages by three signals:**

1. **Position (0-60 points)** — Pages closer to rank 1 have more room to improve
   - Positions 1-3: 10 points (already ranking well)
   - Positions 4-10: 60 points (SWEET SPOT — close to page 1)
   - Positions 11-20: 40 points (still worth optimizing)
   - Positions 21+: 5 points (far down, harder to move)
   - No position data: 0 points

2. **Impressions (0-30 points)** — Pages with real traffic show clearer improvement signal
   - 0 impressions: 0 points
   - 1-20 impressions: 5 points
   - 21-50 impressions: 15 points
   - 51-100 impressions: 25 points
   - 100+ impressions: 30 points

3. **Query Diversity (0-10 bonus points)** — More keywords = more rank-up opportunities
   - Less than 5 queries: 0 points
   - 5-10 queries: 3 points
   - 10-30 queries: 7 points
   - 30+ queries: 10 points

**Total score: 0-100 scale**

**Reason codes (why a page is recommended):**
- `HIGH_PRIORITY` — Positions 4-10 + 100+ impressions (perfect candidates)
- `HIGH_PRIORITY_MED_VIS` — Positions 4-10 + 50-100 impressions (good candidates)
- `MEDIUM_PRIORITY_HIGH_VIS` — Positions 11-20 + 100+ impressions (high traffic but farther from rank 1)
- `MEDIUM_PRIORITY` — Mixed signals, score 50-60 (monitor these)
- `NO_OPPORTUNITY` — Doesn't meet criteria (low score)

**Why this rule beats a fixed cutoff:**
A simple rule like "rank every page at positions 5-15" treats all such pages the same. My rule recognizes that a page at position 7 with 200 impressions is very different from a page at position 7 with 10 impressions. Position + impressions + breadth together predict improvement better than position alone.

In [9]:
# Setup (if not already done)
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib numpy

import duckdb
import pandas as pd
import numpy as np
import os, getpass
import json

# HF token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("✅ Connected to warehouse")

# Build features (from Assignment 4) - CORRECTED
features_query = f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS impressions_prev30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_clicks ELSE 0 END) AS clicks_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_avg_position ELSE NULL END) AS avg_position_prev30,
            AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position ELSE NULL END) AS avg_position_last30,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS impressions_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.ga4_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
    ),
    query_agg AS (
        SELECT
            content_hash_id,
            COUNT(DISTINCT query_hash_id) AS visible_queries
        FROM {TABLES['fact_query_90d']}
        GROUP BY content_hash_id
    ),
    with_queries AS (
        SELECT
            w.content_hash_id, w.client_hash_id, w.impressions_prev30, w.clicks_prev30, w.avg_position_prev30,
            w.impressions_last30, w.avg_position_last30,
            COALESCE(q.visible_queries, 0) AS visible_queries
        FROM windowed w
        LEFT JOIN query_agg q ON w.content_hash_id = q.content_hash_id
    )
    SELECT * FROM with_queries
"""

features_df = con.sql(features_query).df()

# Verify fix - diagnostic check
print(f"\n📊 DIAGNOSTIC CHECK:")
print(f"Unique content_hash_id: {features_df['content_hash_id'].nunique():,}")
print(f"Total rows: {len(features_df):,}")
print(f"Rows per page: {len(features_df) / features_df['content_hash_id'].nunique():.2f}")

if len(features_df) / features_df['content_hash_id'].nunique() > 1.5:
    print(f"⚠️  WARNING: Still have duplicates! Ratio should be ~1.0")
else:
    print(f"✅ FIXED: Proper one-row-per-page aggregation")

# Calculate label: is_improving (both position up AND impressions up)
features_df['is_improving'] = (
    (features_df['avg_position_last30'] < features_df['avg_position_prev30'] - 1.0) &
    (features_df['impressions_last30'] >= 1.1 * features_df['impressions_prev30'])
).astype(int)

print(f"\n✅ Features built: {len(features_df):,} pages")
print(f"Label distribution: {features_df['is_improving'].value_counts().to_dict()}")
print(f"Baseline rate: {100 * features_df['is_improving'].mean():.1f}% of pages improved")


✅ Connected to warehouse


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


📊 DIAGNOSTIC CHECK:
Unique content_hash_id: 90,489
Total rows: 90,489
Rows per page: 1.00
✅ FIXED: Proper one-row-per-page aggregation

✅ Features built: 90,489 pages
Label distribution: {0: 89238, 1: 1251}
Baseline rate: 1.4% of pages improved


## 2. Build the ranked queue

**What I'm doing:**
1. Score each page using the three components (position + impressions + diversity)
2. Assign a reason code based on the profile
3. Sort all pages by score (highest first)
4. Write to CSV so the team can act on the rankings

**Why I write the CSV:**
The notebook generates it fresh on each run — the CSV is never committed to GitHub (blocked by .gitignore). This keeps data out of version control while keeping the notebook reproducible.

In [10]:
# Define the scoring function
def baseline_score(row):
    """
    Score a page by position + impressions + query diversity.
    Returns (score, reason_code).
    """

    # POSITION COMPONENT (0-60)
    if pd.isna(row['avg_position_prev30']) or row['avg_position_prev30'] <= 0:
        pos_score = 0
    elif row['avg_position_prev30'] <= 3:
        pos_score = 10
    elif row['avg_position_prev30'] <= 10:
        pos_score = 60  # SWEET SPOT
    elif row['avg_position_prev30'] <= 20:
        pos_score = 40
    else:
        pos_score = 5

    # IMPRESSIONS COMPONENT (0-30)
    if row['impressions_prev30'] >= 100:
        imp_score = 30
    elif row['impressions_prev30'] >= 51:
        imp_score = 25
    elif row['impressions_prev30'] >= 21:
        imp_score = 15
    elif row['impressions_prev30'] >= 1:
        imp_score = 5
    else:
        imp_score = 0

    # QUERY DIVERSITY BONUS (0-10)
    if row['visible_queries'] >= 30:
        div_score = 10
    elif row['visible_queries'] >= 10:
        div_score = 7
    elif row['visible_queries'] >= 5:
        div_score = 3
    else:
        div_score = 0

    # TOTAL SCORE
    score = pos_score + imp_score + div_score

    # REASON CODE
    if pos_score >= 60 and imp_score >= 25:
        reason = "HIGH_PRIORITY"
    elif pos_score >= 60 and imp_score >= 15:
        reason = "HIGH_PRIORITY_MED_VIS"
    elif pos_score >= 40 and imp_score >= 25:
        reason = "MEDIUM_PRIORITY_HIGH_VIS"
    elif score >= 50:
        reason = "MEDIUM_PRIORITY"
    else:
        reason = "NO_OPPORTUNITY"

    return pd.Series([score, reason])

# Apply scoring to all pages
features_df[['baseline_score', 'reason_code']] = features_df.apply(baseline_score, axis=1)

# Assign action labels based on score
features_df['action'] = pd.cut(
    features_df['baseline_score'],
    bins=[0, 30, 60, 100],
    labels=['MONITOR', 'OPTIMIZE', 'HIGH_PRIORITY']
)

print(f"✅ Baseline score applied to all pages")
print(f"\nScore distribution:")
print(features_df['baseline_score'].describe())
print(f"\nAction distribution:")
print(features_df['action'].value_counts().sort_index(ascending=False))
print(f"\nReason code distribution:")
print(features_df['reason_code'].value_counts())

# Build and rank the queue
ranked_queue = features_df[[
    'content_hash_id', 'client_hash_id',
    'avg_position_prev30', 'impressions_prev30', 'visible_queries',
    'baseline_score', 'reason_code', 'action', 'is_improving'
]].copy()

ranked_queue = ranked_queue.sort_values('baseline_score', ascending=False).reset_index(drop=True)
ranked_queue['rank'] = range(1, len(ranked_queue) + 1)

# Write CSV (will be ignored by .gitignore)
os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"\n✅ Ranked {len(ranked_queue):,} pages")
print(f"✅ Wrote work/outputs/baseline_action_score.csv")
print(f"\nTop 5 preview:")
print(ranked_queue.head(5)[[
    'rank', 'baseline_score', 'action', 'reason_code',
    'avg_position_prev30', 'impressions_prev30', 'is_improving'
]].to_string(index=False))

✅ Baseline score applied to all pages

Score distribution:
count    90489.000000
mean         5.469217
std         14.095067
min          0.000000
25%          0.000000
50%          0.000000
75%          7.000000
max        100.000000
Name: baseline_score, dtype: float64

Action distribution:
action
HIGH_PRIORITY     2185
OPTIMIZE          1507
MONITOR          34548
Name: count, dtype: int64

Reason code distribution:
reason_code
NO_OPPORTUNITY              88082
HIGH_PRIORITY                 953
MEDIUM_PRIORITY_HIGH_VIS      633
MEDIUM_PRIORITY               572
HIGH_PRIORITY_MED_VIS         249
Name: count, dtype: int64

✅ Ranked 90,489 pages
✅ Wrote work/outputs/baseline_action_score.csv

Top 5 preview:
 rank  baseline_score        action   reason_code  avg_position_prev30  impressions_prev30  is_improving
    1             100 HIGH_PRIORITY HIGH_PRIORITY             8.199282               557.0             0
    2             100 HIGH_PRIORITY HIGH_PRIORITY             4.103679   

## 3. Top-20 review

**What I'm checking for each of the top-20 ranked pages:**
1. **Action** — What does my rule recommend?
2. **Reason code** — Why is it in top-20?
3. **Confidence** — Do the signals look solid?
4. **What could make it wrong** — When would this recommendation fail?
5. **Did it actually improve?** — Check the label (ground truth)

**The skeptic's mindset:**
Even if my rule ranks a page high, there might be reasons the ranking doesn't hold:
- Position 7 might be optimal for this query (no room to improve)
- Low impressions means one ranking shift dominates (noise)
- Small number of queries means one query shift = page-level change
- Improvement signal might be random noise, not from optimization

I list concrete skeptical takes for each page so the team knows the limits of my rule.

In [11]:
# Before running features_df, add this diagnostic:
print(f"Unique content_hash_id: {features_df['content_hash_id'].nunique()}")
print(f"Total rows: {len(features_df)}")

# If unique count << total rows, you have duplicates

Unique content_hash_id: 90489
Total rows: 90489


In [12]:
# Deep review of top 20
top_20 = ranked_queue.head(20).copy()

print("=" * 110)
print("TOP-20 SKEPTICAL REVIEW")
print("=" * 110)
print()

for idx, row in top_20.iterrows():
    rank = row['rank']
    score = row['baseline_score']
    action = row['action']
    reason = row['reason_code']
    pos = row['avg_position_prev30']
    imps = row['impressions_prev30']
    queries = row['visible_queries']
    actually_improved = row['is_improving']

    print(f"RANK {rank:2d} | Score {score:3.0f} | Action: {action:15s} | {reason}")
    print(f"         Position: {pos:5.1f} | Impressions: {imps:6.0f} | Queries: {queries:3.0f}")
    print(f"         ✓ Actually improved: {'YES ✅' if actually_improved == 1 else 'NO ❌'}")

    # Build skeptic's list
    skeptics = []

    if pos >= 4 and pos <= 10:
        skeptics.append("Position 4-10 might already be optimal — query intent match, not improvable")

    if imps < 50:
        skeptics.append("Low traffic: one ranking shift = big % change. Noise, not real improvement")

    if queries < 5:
        skeptics.append("Too few queries: single query shift dominates. Not representative of page quality")

    if pos > 15:
        skeptics.append("Far from top 10: algorithm changes might matter more than content optimization")

    if imps > 0 and (imps / max(queries, 1)) < 5:
        skeptics.append("Very few impressions per query: these keywords might have low search volume")

    if skeptics:
        print(f"         ⚠️  What could make this WRONG:")
        for i, skep in enumerate(skeptics, 1):
            print(f"            {i}. {skep}")
    else:
        print(f"         ✓ Looks solid: good position + good visibility + broad keywords")

    print()

# Calculate Precision@20 (baseline metric)
top_20_improved = top_20['is_improving'].sum()
precision_20 = top_20_improved / len(top_20)

print("\n" + "=" * 110)
print("BASELINE PERFORMANCE: Precision@20")
print("=" * 110)
print(f"Top 20 recommended pages: 20")
print(f"Pages that actually improved: {top_20_improved}")
print(f"Precision@20: {precision_20:.1%}")
print()
print(f"Meaning:")
print(f"  My hand-written rule identifies {top_20_improved} out of 20 correct optimizations in the top-20.")
print(f"  Week 5 target: ML model must beat {precision_20:.1%} Precision@20 to justify its complexity.")

TOP-20 SKEPTICAL REVIEW

RANK  1 | Score 100 | Action: HIGH_PRIORITY   | HIGH_PRIORITY
         Position:   8.2 | Impressions:    557 | Queries:  57
         ✓ Actually improved: NO ❌
         ⚠️  What could make this WRONG:
            1. Position 4-10 might already be optimal — query intent match, not improvable

RANK  2 | Score 100 | Action: HIGH_PRIORITY   | HIGH_PRIORITY
         Position:   4.1 | Impressions:    299 | Queries: 145
         ✓ Actually improved: NO ❌
         ⚠️  What could make this WRONG:
            1. Position 4-10 might already be optimal — query intent match, not improvable
            2. Very few impressions per query: these keywords might have low search volume

RANK  3 | Score 100 | Action: HIGH_PRIORITY   | HIGH_PRIORITY
         Position:   7.1 | Impressions:    218 | Queries:  71
         ✓ Actually improved: NO ❌
         ⚠️  What could make this WRONG:
            1. Position 4-10 might already be optimal — query intent match, not improvable
         

## 4. Weak picks + leakage check

**What makes a "weak pick":**
A page my rule ranks high but shouldn't be in top-20 because:
- Position signal is misleading (already optimal, no room to improve)
- Impressions too low (noise dominates)
- Query diversity too low (one query dominates)
- Actually DIDN'T improve (label says no) — rule failed

**Leakage check (critical):**
I must verify my features come ONLY from the feature window (prev_30), not the label window (last_30):
- ✅ `avg_position_prev30`, `impressions_prev30` — from BEFORE the outcome
- ✅ `visible_queries` — from 90-day snapshot, known before prediction
- ❌ NEVER use `avg_position_last30`, `impressions_last30` as features — these ARE the outcome
- ❌ NEVER use `trend_direction` or `trend_pct` — encode the label already

No leakage = rule uses honest historical signals only.

In [13]:
# Find weak picks (high score but didn't improve)
print("=" * 110)
print("WEAK PICKS: Pages ranked high but didn't actually improve")
print("=" * 110)
print()

weak_picks = ranked_queue[(ranked_queue['baseline_score'] >= 60) & (ranked_queue['is_improving'] == 0)]

if len(weak_picks) > 0:
    print(f"Found {len(weak_picks)} weak picks in top scorers (score >= 60):\n")

    for idx, row in weak_picks.head(10).iterrows():
        rank = row['rank']
        score = row['baseline_score']
        reason = row['reason_code']
        pos = row['avg_position_prev30']
        imps = row['impressions_prev30']
        queries = row['visible_queries']

        print(f"Rank {rank:3d} | Score {score:3.0f} | {reason:25s} | Pos={pos:5.1f}, Imps={imps:6.0f}, Q={queries:3.0f}")

        # Diagnose why it failed
        if imps < 30:
            print(f"          → Low impressions ({imps:.0f}): likely noise or bad luck")
        if pos > 15:
            print(f"          → Position {pos:.1f}: algorithm shifts dominate content changes")
        if queries < 3:
            print(f"          → Only {queries:.0f} queries: one query shift = page-level")
        print()
else:
    print("✅ No weak picks: all high-scoring pages actually improved!")
    print()

# Leakage verification
print("\n" + "=" * 110)
print("LEAKAGE CHECK: Verify no future-window features used")
print("=" * 110)
print()

# Check what features are in the scoring function
leakage_check = {
    'avg_position_prev30': '✅ Feature window (prev 30 days)',
    'impressions_prev30': '✅ Feature window (prev 30 days)',
    'visible_queries': '✅ From 90-day snapshot (before prediction)',
    'avg_position_last30': '❌ LABEL WINDOW — not used in scoring',
    'impressions_last30': '❌ LABEL WINDOW — not used in scoring',
    'trend_direction': '❌ ENCODES LABEL — not used',
    'is_improving': '❌ LABEL ITSELF — not used in scoring'
}

for feature, status in leakage_check.items():
    print(f"{status} | {feature}")

print()
print("✅ LEAKAGE VERDICT: No future-window features in baseline_score calculation.")
print("✅ Rule uses only historical signals known at decision time.")

WEAK PICKS: Pages ranked high but didn't actually improve

Found 1785 weak picks in top scorers (score >= 60):

Rank   1 | Score 100 | HIGH_PRIORITY             | Pos=  8.2, Imps=   557, Q= 57

Rank   2 | Score 100 | HIGH_PRIORITY             | Pos=  4.1, Imps=   299, Q=145

Rank   3 | Score 100 | HIGH_PRIORITY             | Pos=  7.1, Imps=   218, Q= 71

Rank   4 | Score 100 | HIGH_PRIORITY             | Pos=  5.5, Imps=   276, Q= 73

Rank   5 | Score 100 | HIGH_PRIORITY             | Pos=  9.3, Imps=   107, Q=319

Rank   6 | Score 100 | HIGH_PRIORITY             | Pos=  5.8, Imps=   543, Q= 99

Rank   7 | Score 100 | HIGH_PRIORITY             | Pos=  4.8, Imps=   152, Q= 93

Rank   8 | Score 100 | HIGH_PRIORITY             | Pos=  3.7, Imps=   171, Q= 35

Rank   9 | Score 100 | HIGH_PRIORITY             | Pos=  7.0, Imps=   952, Q=156

Rank  10 | Score 100 | HIGH_PRIORITY             | Pos=  5.9, Imps=   137, Q= 34


LEAKAGE CHECK: Verify no future-window features used

✅ Feature win

## Baseline Rule Assessment

**Precision@20: 0%**

The hand-written rule targets already-successful pages (positions 4-10, high visibility).
These pages show almost no improvement because they're already optimized.

The rule design was flawed, not the implementation. This honest baseline (0%) is what
the Week-5 ML model must beat to be justified.

Next: Build ML model to find pages with actual improvement potential.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.